In [48]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt

In [49]:
df = pd.read_csv(r"C:\Users\Junayed\pandas_prac\Aug_23\messy_trainers.csv")

In [50]:
df.head(5)

,TrainerID,TrainerName,Specialty,City,YearsExperience,SessionRate,SessionsThisMonth,ClientRating,JoinDate,IsCertified,Notes
0,T001,Marcus Reyes,Pilates,Portland,17.0,92.06,28.0,3.8,2023-06-16,No,NaN
1,T002,Elena Voss,Pilates,Boise,8.0,61.53,89.0,3.4,2023-08-12,Yes,NaN
2,T003,Rico Callahan,yoga,Denver,2.0,42.62,35.0,3.3,2023-11-27,No,NaN
3,T004,Dana Marsh,Pilates,Tulsa,3.0,36.9,17.0,3.9,2023-11-17,Yes,NaN
4,T005,Owen Iqbal,Yoga,Nashville,15.0,$78.50,64.0,4.2,2023-09-11,Yes,NaN


## Specialty Case Harmonization

**Issue Identified:**  
The `Specialty` column contains disparate casing formats (e.g., `"STRENGTH"`, `"Strength"`, `"yoga"`, `"Yoga"`, `"Crossfit"`, `"CrossFit"`) and potential whitespace variations, leading to fragmented category buckets during aggregation and analysis.

**Fix Applied:**  
1. **Whitespace Trimming:** Applied `.str.strip()` to remove leading and trailing spaces.
2. **Title-Case Normalization:** Standardized all category labels to uniform title case using `.str.title()`, consolidating identical specialties into clean, distinct categories (e.g., `Pilates`, `Yoga`, `Strength`, `Cardio`, `Crossfit`).

In [51]:
df["Specialty"] = df["Specialty"].str.strip().str.title()

In [52]:
df["SessionRate"] = pd.to_numeric(
    df["SessionRate"].astype(str).str.replace(r"[$,]", "", regex=True),
    errors="coerce"
)

In [53]:
certificate_map = {
    # Variations of Yes
    "yes": "Yes",
    "Yes": "Yes",
    "YES": "Yes",
    "Y": "Yes",
    "y": "Yes",
    "YYes": "Yes",
    "1": "Yes",
    1: "Yes",
    True: "Yes",
    
    # Variations of No
    "no": "No",
    "No": "No",
    "NO": "No",
    "N": "No",
    "n": "No",
    "0": "No",
    0: "No",
    False: "No",
}

df["IsCertified"] = df["IsCertified"].astype(str).str.strip().map(certificate_map)

In [59]:
df["JoinDate"] = pd.to_datetime(df["JoinDate"], format='mixed')
df["JoinMonth"] = df["JoinDate"].dt.month_name()

df[["JoinDate", "JoinMonth"]].head(5)

,JoinDate,JoinMonth
0,2023-06-16,June
1,2023-08-12,August
2,2023-11-27,November
3,2023-11-17,November
4,2023-09-11,September


# EDA

In [54]:
df.groupby("Specialty")["ClientRating"].mean().sort_values(ascending=False) 

Specialty
Strength    3.887500
Yoga        3.828571
Cardio      3.812500
Pilates     3.744444
Crossfit    3.714286
Name: ClientRating, dtype: float64

In [55]:
df.groupby("Specialty")["YearsExperience"].mean().sort_values(ascending=False)

Specialty
Yoga        12.571429
Cardio      11.875000
Strength    11.750000
Crossfit    11.125000
Pilates      8.500000
Name: YearsExperience, dtype: float64

In [56]:
df.loc[df["YearsExperience"] > 15, ["TrainerID","TrainerName"]].reset_index(drop=True).to_csv("15_years_experienced_trainers.csv", index=False)

In [57]:
df.loc[(df["YearsExperience"] > 10) & (df["IsCertified"] == "Yes") & (df["Specialty"] == "Yoga") & (df["ClientRating"] > 3) & (df["SessionRate"] >10), ["TrainerID","TrainerName", "Specialty", "IsCertified", "ClientRating", "SessionRate"]].reset_index(drop=True)

,TrainerID,TrainerName,Specialty,IsCertified,ClientRating,SessionRate
0,T005,Owen Iqbal,Yoga,Yes,4.2,78.50
1,T022,Talia Fontaine,Yoga,Yes,4.1,87.80
2,T031,Drew Khan,Yoga,Yes,3.7,92.53
